# EODHD Bollinger — Colab
Önce MK_BB_EODHD_Colab_Netlify.zip dosyasını yükleyin. EODHD_API_TOKEN değerini Colab Secrets'a ekleyin. Bu notebook Netlify'a kendiliğinden yayın yapmaz.


In [ ]:
from google.colab import files, userdata
import os, zipfile
from pathlib import Path
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/mk_bb_eodhd')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        dest = (root/member.filename).resolve()
        if not dest.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive path')
        if member.filename in ('config.json','universe.json') and dest.exists():
            print('Preserved:', member.filename)
            continue
        z.extract(member, root)
os.chdir(root)
os.environ['EODHD_API_TOKEN'] = userdata.get('EODHD_API_TOKEN')
%pip -q install -r requirements.txt


In [ ]:
!python -m unittest -v test_engine
!python bb_eodhd.py --init
!python bb_eodhd.py --catalog


## Sembol doğrulama — v2.1
private/catalog.json gerçek INDX ve FOREX kayıtlarını içerir. COMM sorgulanmaz. Eşleşmeyen endeksler için universe.json ayarlarını kontrol edin. USD spot metal çiftleri katalogda varsa doğrulanır. Emtia date/value serileri commodities.py ile yerel frekansta çizilir; OHLCV üretilemez, stoplu backtest kapalıdır. Soybeans/Cocoa desteklenmeyen slotlar olarak görünür.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'bb_eodhd.py'], check=True)
files.download('netlify_site.zip')


Günlük otomasyon için kaynak paketteki .github/workflows/daily.yml dosyasını özel GitHub reposuna yükleyin. EODHD_API_TOKEN, NETLIFY_AUTH_TOKEN ve NETLIFY_SITE_ID Actions Secrets olarak tanımlanmalı. İlk başarılı build, erişim ve lisans kontrolünden sonra PUBLISH_APPROVED repository variable değerini true yapın. Ayrıntılar README_TR.md.
